# Furusato Workshop 3.0.0 — Notebook_04_Furusato_Provision_Complete_Workshop

**Workshop version / 教材版:** 3.0.0

**Runtime profile:** v3.0.0-preview. **Data contract:** 2.7.0-realistic.1 (unchanged CSVs).

**Production placement:** the specified folder directly; no Temp folder is required.

Run one explicit stage at a time. Default is read-only preflight. Native Activator start/stop and correlated complete-file delivery remain separate gates. Run Notebook05 only after the three increments; it reads Lakehouse/Files, not Eventhouse.

Use the same private resource plan, gate and deployment-state evidence across stages. Environment identities are parameters, never embedded in this notebook. CLI deployment is the primary cross-machine orchestration path; this frontend uses normal NotebookUtils authentication. The sealed package includes the source-grounded, complete-contract, time-layer-isolation and standard-contract-restoration compilers, their SQL views and query examples. Apply the corrected profile only to verified fresh sources; the historical four-source baseline alone is not the corrected Agent.


In [ ]:
# Fabric parameter cell — environment identities remain in private JSON files.
PARTICIPANT_ID = "001"
ENVIRONMENT = ""  # Explicitly choose dev, test or prod.
EXPECTED_WORKSPACE_NAME = ""
SCOPE_FILE = ""  # Private scope JSON; use notebookutils.nbResPath + "/builtin/scope.json".
PLAN_FILE = ""  # Private resource-plan.json from preflight.
WRITE_GATE_FILE = ""  # Explicit folder mapping/write approval; never generated as approved.
PRIVATE_EVIDENCE_DIR = ""  # Outside the extracted source package; preserve between stages.
ACTION = "preflight"
APPLY_CHANGES = False
ALLOW_AUTOMATED_APPLY = False
CONFIRMED_PLAN_SHA256 = ""
RUN_NOTEBOOK01 = False
EXPECTED_DEFINITION_SHA256 = ""
RULE_STATEMENT_OVERRIDES = {}


In [ ]:
import base64, gzip, hashlib, json, sys, tempfile
from pathlib import Path
_PAYLOAD_CHUNKS = []


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("+D2WY386753fp7brL8XZ0mdoNdm0T8R9gYFqy7t/02e/PuiRjI6YcR5CMxKpRlp6//Seqe6/+7ZaD80nkfuqVvWGauqNptG/m4+O+XRwt5ndFcym0XA6erPfNALlofMlxUaTEXTYLX5+Bw3+hp+b5Hd5QD98rQOMAtuD/gMjM9mr1l5+F/4A53kcFudMPRM5BdBf75CLqf7Ky7NTeRUuhreULfj8OaZvyqwH0NDDnhaXc/kWtFgSspNO4Z9tj7xBPBQuvkfB/GEU88Qx/Z+hd6Noqt9r9ykmX1jpvmnhrT8N555ZeDdtyl3329fvw+pdsdoRMfnjRqotRVIkB7aDsVL0iWffFQpa3ZwDBxQU3V/BpnAd9grwGiVVr7oH/TL98by2xsBQoEWtHky12gP0zM8v1DPUN8PJw5T0A5z1xerh79Vn0LwHPeR+OaIvwRfATFPwwPqc3kqx2vtxmXh+vIb4OXiPvY9gb0OmOqVCX1NdL5qa2ez8MItZWh5fOIuK160pnhLnWKQxmlbsL2sDM9+V11/ccGasenexj+vI99Mo2lGlpOg9uas7czXqrdR6b+dwHqSnBl3L3imSLWv1RmTrqqv09vR8JB9LQi6f0efB48MCmIl0/7wMh/ricUkZzZ/ft4KPR5IO+XAPvvwu5MYojGRHut3bfQKXnsgO+emLNZkfaHbukacb/Y7pB42crEhonrLCOqaB5h9H47ZkFBQvYCvCleMFgeoC60ZOIGLdemPlEOd6quewwDbqh6c0HlbX2vD7ArRDOvcEzSU2kDbFe1ijJ9K5ZvX4WahZHdgi8kobjhdfQsuP6TF532nJ+4I9ErSe3uMc/2hVT7N+m09BM9MMf+TotrI4kSf7Z3MtyfPpVwU994xt3TS2VbNhPBz2P1mPWaB/rKpTr5qdjl7oQ/+qhmWYVcOFfl1OA9CIDf2a2Hzzyftxaz0M7e7i6dtz4HZEHtpRbVnVVaRGpEEGgf5lBfZrV+/PlXpjZXvjslor4DxZDD0JlagTaJbNGCh44ffZV7tVlh9+TkETK0HDy939V2eoCxsq47/g/3/tfjHkr3e/y45WlZfSnDxAjCR/G67kh101wUj3CU7CT62a9TUkdpyHfx9jvUadlsVaoY/fafvcH58T/vaIvyX463sXPyX8QD9/d094rHb/MP/6GD49PvvPzo8YXxF3/+RPe/a9Ziw7u5d57hW46q5vFH+Yvtm6M7YG6ED/nnu2ssCZX5J/DGQFF3n05xfjK4TVWf7U+z54bwtdtFItM3DAn4rVpi0aKd4MZ973lfp4B3y+VfRGac+f+ZiHwFfULytN8A95qe5OIWipX3aVorrbMB4LDDpsifPpDkEHrTb3P2OnPpLOQT9aejbtRhDdtYLFsOnMR7cmZJ7g1/pP8Cx1F/Tb/Zp+A/Cp92Wv0yoz/2Yj/E5h86kt9CT10XDqa1MDskJZi/+PeXsGPeWuhY6rhOvp/XIarz95D3HN4+G6WI9RD3450WXLn7JHPXBWp52XAXsZ/NRurJp9s5o9/wwuCR9Dyvdp+r3MPjQLvL9hBGbze81/J26r++vu88Pyodk1Gv45ueBrrUZBbdkF7a4Q2RbnJPcDrW6vRHu9WhGyoF3W6LP3xpKqV/Zy4XC+4O+uW5Ef7n7wrL9+h1K13SA9a8iOytfvLfD291XMz4IX9zlK6X2E//Ny3tNyXJgvHdB+XKoQRPrCfBrF/mfOfgYdxY08T0qGLAXmEZvnzoDHFEndFQqONyuyZEjl36JOqFh9T2v18H7KVuRElcwVeHEHbRL0FhwZLsfjxK/M7KPPNs6ZivdHSbDaQU8l2HNK3xr11HD8qFktYW/EOG00X7sPrqB/0ubQ9tZTH3xQfwA9rEEbDwl+89d3N89fSPOxby6LwfY5ZdrRc6/Nybmyny/hh5P9DtsFO1RklmjZeifkfiutjtvVA4/7bUdVyJ6K7Oh+Cf9GZ/f7Vi2Oei/OoMtOUT5Dly1FlsOvsnx3n+od0+zXhd5dhjXhG/wOPPbtbvP1+8/f5OWXb7j24cim29PpHq/GZ+fM3KppNpTFaW6d8CUbwySHGzzZHlqQx3EuNOhYLo4Y+w23PIMQdmbknMWvPdZ4FzR9XOrqjS1tB7VedTkryvHmQTwk2AR/KmUlsndOnHOMtRbnkA1fHaMj4/7reJz7tdl6mXmV6R6KPdo9yT9dR378acX7Rf1615EfvozS/dzrRFzP/UoxTWybCj0+x08U62JtEf/e4h6vgSFcufslvm9X+i3Z+5wP5Zyc4DP+uD1XfU1veko0d7uUG7qBPbdlp1ac25bpd3XoS85UbbUlrT7bqAeeOjTwgNwyaXf33nse6axD44Sm03NIMRl+J15SxFk8tr7Kj25bXg5l+ZE46mj/jj6/Wl/wFvnacnw78stOvQedYUS0dx3wP+Rr3LtA6pUUryLZemVnww57s3xtVudxPkSfPoiHkXSQsWlpoiEB42giZnHeRzuNZeSCcnVXPe8bEd9/wS8k5HTejn5cTx8e1+F8EdvFAoMsp9fkcu77tedp/Pv6el3JW+QG5OokUCyjBHsswv6v1FbTg41Whn4vk4Ztz2GzeM+pKwVl9nY6pY429s2z/1bY8U/FiIczvYLjTmtEuEdPoMH1h2RVSykpVmOnho2dmO1sNXDOhkz7TvXau67FUQHGBnxZdLzK1n67POoIzCXNLayh6Jz6RwA+B9fjFyPhEwyWndFy9hW//0x8x8KHl9b0ZO9nijjvNZyCs/dZNpvsncirELkGzDHBmY/jXPPj3iol1YNsb7U36q4YqN4YtlCvoEbKStMBV4WPrldyoGfxt53jNYQswj7ADu8sp6FZo1wfhTdrp3d11lrcFCq/T8Hc9rL25mfsWeZ+6rk9O5Id78PO9ItBRshq3QeWawYK9kmVFMqVDWT+invGWZW2zn9Zbv9W7Hy+PXPsK8rE96dCZgp/DrBw/Qm/r/A7+JO/V9cxn/vBOqwDE9ceY5l6xKOZ+xnMY7hcE/F+/KAUnZaCPeqEXb1dUqIxeNKZKzvgB4gFYOCAtSEK5C6w8FZ9O2a7WJuRysxPqhk79EtZKOf7mFCGVd5Y4nw5T/EPzXf4r//tf59PecikMum3/vrxIUklAIQdtabyQ21B15RMW7or/fhqazeAXLo8+jmW5Z0iPturqJPUg3WaTvC1uy4L8/jhS0/ufh2dwNvFudSq09QCkFQZ8GmjegbEFcQ63TnQlCQnUVqEDVTCDli1vVX1WXRgocx9p8JsLM+0YYPpFD+/uKtQi6EKIErN/XKJbTL3eCvbFM31cNAsxqwC0S2Z8hvFdhkwu6jVxzLeWbK9mSzEtteGOKqy3bKkRBMfEKegeICAnv8pYttve6HbqWfDnv66/ZgN61P14brSFVXGzwtXy1/fsA9QTxCzFt59VyjiX8K4nVOnCDZ2jlucQ5xAJPcimCxFtdV77z7s2y0n9NPz+CNo50AvhEWeoBl3/lOjiyJ2RRxcHaQ38b3LtDRNv4vrPg36QkhZvUjRWXbX3tki5ONvbeyPrTdED1DbMjaqBVVmtSV79i4T7cx447P8xv2QV1p480CIud6HdSpL7QAtf67p4kyvs8r4W22uiZCxSL06wFfy7KvcPvG9Zu9z95zKnp7s0F3TMiBn7A2hkh0Kd05g6/5KiYIQ0HOH6yTF60Xvc5/d7Hqg9Sv72HO1YdvHS6y+DAPIqRphOkORNC9+rk9UfXzdZ9EV96DVCZX6OOpaRsR5mGrddNVaca5EagBTV1KjYK7oHd/W++47TX+OCPbskrlJStlOzP84zNJSUr2UmkRn9FNt7zKwrfbp58IF8yzLIoXhZxKePO92uaADP8t9YOvtCHDJUzy6C40y3YUaeLZrdeZOKPr1hpDvZa3V2yowWffPNQ49SkUZP9M1jZsd054nWIedlJy+RQZebo2378Z0FnLxXD4Rdl0aZyzS9d/aOi5bxtbLpJB3Np+edp1Arv/yP/1v/+k//s/n8RZdp+TnsPLwZSqF6zS19BC2vQLzGd51ZmsR/q1BhvJvlAHkf7rHa8l1tbVIRY3Dxe6FUPBlWbpMU9czacLH8qC9U+tkLQVmpOlqept6ZqdBHqhS3+vqTVeN+vixIyVqlOhiOUOzibsykXnNwLFMWAdGvyGPJFce7X7E4dbUpXohfHL0+Ufw2Wt5Fvhia4cNiamuwGIlxWVIxSh19Rn/XalW37WjcQTO3cJM3L1DJ3Cfus7Af8kljf35Gabu53P78gpXsQkTdDuxjEQnrILpG/fDBvYUZl7EdjXKVqWOjIjP54EiQYZJDWLYDTB7CXhWuiDD9mt6+14dXMVfv1u/iT0ZhYVTWR4WXqCPV7tTKBsdr+krtWLo1OmC6kWqPgYebZeJy/G+RTUygN/72IO341GL4ddDilkY/hwF65EuMHgoPwee/RguOiJNuq+A32m/uGYjSdE7em9nPZJt6TtTFz5LbznQEapkbHi+ttcAxiwUVKstd63mXK0zbNAoKzhrBzoNPFB4u9u7WaSuEW2S9iHF5D31xi5JxWhDh/+U7S+lOPyC/3dqapyi6Ib4V5aX699it+orwyyZe/4hLpKWUlT1CfaILspe5Ogz7JUaarVC2faqvpAhuhJpetVV9InrvB2jq/isMGxgPVYnyNJQkkr48HP5jXQULtrew8PQWyy7j0zL5v8vguFj4H27Xz6ItPdnf/0d/44WYfjtNw+MfpRuvbf78N3ZZ9l6kupVZGDpEuRIEXjSo62nhrTzgd/dogslATu/slVbbVnR7Tfzlm7drCetucAVd2Ejob9OobcIsiUp0J1T8+fa+rHa7NNvqaOcLkzMnI6CXCHdnJM3/Ptn8Zymz2Bj2AXBX2FjK9yR9Rl4ruMCH2J/5qAriAa9vVGlXvHtPAebZtCJ6edkbvYeD3zSfNFEbv+bda391//hf/8v/+d//E//9384D/WspqtlKsei2yD8+oMZT3tXWz7zq/XwgsnCjLA4OpRzr52r0sk898xwSYicMlRSwQEZKXVDdkBGSr23hejZ0GMLUorUVsezIwNiyC5mPNKH+355wZ0WjcIH+/YP8d6bAD/jgZnClxgmvUkEg4d1Zh+zOskMbW8METzbqm5BViRTwFdFcjzsicwJZ+8RwWenlJ1GRdJsb7rbchl5Z1xrR59/lmutOnfqNlR0pQwVvlXrzLYgfPUhTtpl7a5QUlrA5vp4Z1v4t9XYP/cwLsXnMMhUHLxNFF+atBbv1emeDL3l7FtpOcP/r5fRYtGO4r8vuwvYz6zc4n56s4ff4iot57dUxb20p5nrPm1vfQeqHSoKPFaca1BZAibW6cZWtvZdYavpvaLiVcow9Rl5+/D+9XPdnDKwOC9rFPkhrNDV/xoXSuJ2WV8OB7zBnXJFxn2ammz1SrZX2TgWzJGw77ESUfHaEeTbDjKNbpZICU0fZskGXF78Q10qr3CjZM/mk9wor5u49m9Dzf4//+P/+7/+X//5P/x3eTWbCWbqtyyuP6jVbBJ2bA2APL9DfWpMtIBVqP3MqdM9SaZFsK+JWOULvU+SguqzsuPNqFZXitf3NSYF6Sa9xq6Kvzn19sYJHd9pKRtFr84vBH3fGbH6QKD3wxGrqq/oHVeJejug052ijyWIPJkeIqD3jXZX9FXLCbS6X1YtZeO02m+P1Oyb+h2iNIl3PLN356IvR59/WtQFwlv3i6LQxrKLTKqAiN+pu6Kr1NuFrqUw8QhwS/X4/++AEyLonVeN1U5fyxYivRipyrx7JvJ0dY/+iAjVbEeIoUiEXjOJBTRqHWLYLc4dfRJ2AclUSZEhcuZaXfWddyYIXFSDGdkQF7vX2vKD2zuVD3m1dfz5pyUPWj2IVyOyPWUF09JzarDoYBl1rXbk1Cu08spiIJIehI7efocnMTMHPqWfvFWXbVDySZbdZOmIOQl/C5XD8VTP03+JJyo+/cv4YbF6HI5X/+o9PSz2/ds6aWtE0UbuEDO4uc/GDLrBSS+2tK9ahGtFKx3x97h9XT4WEbe6cyE5vUkzbZkorhNxjW54SKMf75KebOxUbiVtbtL2ngOFPdhG+zZ+oqWeHMdN3ENbwOTvx0Dg0DYvnDyPWr113ML05HsEv0F7hmubhWQthxZD7A9nJf3h9n3i2Iowpnixh/t0nH2LP95DUIj4/GKrhNz1/fm0Jvbo5VKzq2UmF1JOTVFiMs6X7PfHLN3Or+Oo7GTHNXX3ZQBin2r7louz9u3T4bspYmC/vTQ/4LDfpB0xZif+PE3fPS6Xzl2Ps5mJPfwj2xgMap1c+40BS6fvcus4LtMTa0rKcdz4uupNSm/vpZMXy/v/Eno5Xx7/2j3a/DTi0l7RnpL0kvBYbd+ek99N4nCC1tK1HfiXrT33rTK7afuStGXJcalW7nvQXGVBb/4fWILf3LgGNP/oVsiwzihUIdVFGvsr+at9vEds5TlJ5W2i6X5MJDnqJ++DPS46adtTSS5Arj47JY7rvvlJjbLvexmK1D4JRnlxeFeEFmoWKaud2l7uEulUgVo4jjsYD0y2SoxSzXTSZjQrz2/7u4llJD01HaZ1+km5TmxQxjL/GehfIOcu1jS69Vds4cu2Zrh/kSUp7dnLvTUfH55d9nh0F7Pf2DAzcBfTf5kPn+bTpyP1hm01A0OazxXPkOLWn9VA0ec+0w8crzq3PTVUvJkEcOKqnDLhKTtQwpZhXgWq1o5mRaeluqy2UXV/p0b4sXolTTe2Wsv08DNXw3aqDsOhxY563I6n7wexBWBT6IMlm3rf7K1Hop1ohcZCwWn1Q6VulxyrUQSO3dn6DAZSQwIonCuW47JzkBrZeF6v5HhNT9MZRqns8DtA9hw/RkGt9z1VBzht4Z2yqpBA7e5Gv2s0dhBmumkU9e6iX54IUdHG+7VLdoT7eD5AemWnAayrXgWgbCZp9TbAF4wVhmXqE19pqXPHs4tOfSwrdYh2qVdSI2ZWN0r4Pq7lmOWmn+1sK95bEiQVOYMewdt6yA5apd46bv8qSCJUQidwPCgU3YaxWNlq9UmotUBWXk+yIxiMek9SolnZjgJoiN5W0/G+zKStV+eKbuIsx/iuUtb0sezU/a2qG8Uz69jDDJCbRLE13hX3vU4P68E6omaggBUghgBblAjnK6m6jfvDkKsbBKhlRfgOZzLO31VY/e6ZIc6tqEj2xqk7rLDewfCTVct0bc+XT9eT7c6Rm1LIvqnrya16r9Qr226oRpN4n2jnbBTQodZi5nVjo0Smr3i9MgyGAM8rKFIbdGRI9LfZkhOo9VmJFY4wRkGrTV+jAQJDzQ57ZadlwHBT3ruuXWZdeE4Hhq0igXdg8DmeEo03jgf6FRVCTRrGoM0enteWnBB0VK8Uca40gsqq1MM5VjZMc3SY0hgqWydsR+9cV5RZl2uDT0Fbnuo1QEd2GXweaZaxVbxKSas3YLAGnsJQtsTUQWWjevMQZ1zG+YU23sYOwd8cH15vROCVHXijfLKukuh29GwvmEIkMzUn7mB/q2C9aiJzbBo6RY1dEchXLaMAWgdN+WXQbokpZDhDyIN2AdKooEqqr7Z6O6XecVV2W7DM0A5xbla7qLZAYeBJ1cuoRCG2G89jVkPVblxAeBqfEf4/sx9tPNOI8H08r419b7oiHgK5p0RKQamrczsCTdXHoG2/pOgN0LddgGkAemKIv0lDjPKhiBOWNcuG7PBLF/cj9ZGHAeQhoELJFjJPjeauHRklpcVnYt9b4HGvIatRlc4RGevyaRBq9WaAdyzg/MDfyhbP5Y8Y2Y49ce0Q66pPAjtUduqpzIsI3bq5yZw3O1ZGZ9cCuQX+7EDmk7dtGTJMUsjjulhL0bFsyKJGwcZzQZ+gG6ZazyJV6rtM9yOt4Fwhn+ZzVZ94MHF85+pabnLPd3SDZ0H/dwRZCvnR3oLe6BeHjHECWyLNKZStO1VScDadOVPCQSeygmfZOBdWikJPhJDPodpyXGV36fk0Tvs5emCMSxFywZCdVqPISg7VM+f4OyAG9sWyJchhX623I5HGInVcpcVuETOJaS84gw0MdfA6q/wgY+q2fIkeRnl/PVt2Z/hj7lEX4pk+dJ0EOQbag+zkGbdYyQb5obe3CrSbIhng2xnXK1J+oY+LWH8ZugI6kXzW9zWrCRpRLvFHbohbjh5a7aIS4VxbStFhmIqVVB72BjIT+EHmeatRBetS8Hxb5hmJs8Ee8jrbA+SRqK8YRWuU1TpWdHIexbmAV7Wb5N/icrRQ2eHYS6ouIMvmhRiugdZbjNO1wbd9n508uEYn7MzBxxumTdqWvVFbBmiCshb6uF4NqROdekBsAD3Qk+2I68Ee6Q2cY1ZurLajWq7TfFPI0lbcoXlQZGt0czMo2buuN14rNflxDNNf8DFll6XsQA8b8Az07xwyjBNPIOeAZRxLiewQ8qpuFHGiTL8ucLK1qptuTO9N8FgHtAa5C50KnS1lu/uP9h30V1G3cKgwHESVba7zcFF04J7jTO/1A/S9F52F7+Tl+BZ6I6Z3GfKd8seziam8PuSeAdqC3NVt6IUGzg/6gil8ddBuaGxVyEbbalLHFsBnMvbVY/cmERv12mXitAy9r/cdykrK82ECwAp6qXOfc61wItNtkFuvfphWsB4OqvIImFLICItnNyswxQGyGf/2sWcO1jSDDIe+B0ZyICuF3rSwj1GMN1S9sXX0ZgA9C3nNNAloXsh8mz+7V+1zlO/wTDOGnVHZvbS4GbVM/v4DZs0zpzpM83sNegVmA2bE2W5syHPSBeQW6MMuaPUKdH4P+rZRJu61JRu6rhqqYY94BVjGB21X505IrFedQ0bu1DwmuLLX/dxeJzEs4WwbsNPtoPrkDPpBOgl42LrZdMNgM7VkN+a5vqsylYv8HILHwjZ0HbFLH3iUfEVsY+wU2BG2BGwFTAWdACwGPRH2oKPFPodqfYyzqABLzPDOjQzPHfZ4vLs5dLUr/r5RL059UO+NgRlXypk381HYv+/5fo5OhG6yemWuwxE6HHYO4/ih0PNbheMVIuAf2A7AW8Ad4E/gdhXvpUT+hlgA9ARcTN0MvvVwZhlddjAr/efM7yuHDt9aETrbAeaIp5l1OdCRXfHxbzqGIbZ3ZgWtZRcU6HroTRn6ZsPSAbVOHoOt4SlbO8TeAftAVjDdfiNwNXAd1lsS6/VUF/gDmASGJK7X7l5e43hvtiqrUZbPDjYI7Ao6nIH9PB9YFDin7heYB6DAlKduE2evt4FP8ZkOfK/7JcgI4BXIjhZsoZYh8Z2gC0LoqJ0dNnYZen0+8AlxiP1sFOZV2qTsWGfs5ML0NrYJcU+ciwmdy9IfA/pmDNyhgqeBSYHtYcXNlRA8701cfsZrISOwp2OcH3USZC3T5us4R6ZqUofWzu/ROOyHXenA412RHnbzNJImcje8KY1ilxLe2wQ9U8f2Q9jNwCsN0BXsUKwBewb5DxuYwYv6DHLSAJZi4672BlhmC41c4rlCT5eJZzUL/6dXCtf2xjmkrq8y611zvQmPBsAmBU6EFOm3kMmKHoTAlRIrMLF3kN0N4NMA+rAHGp+xKwnkewW6kNhgLDE9W2PeD/gb+mmr5PRiZh9KeH/3Jtzz5a6YWa9cmLRifahYlAkmeK0HuQu5DAyhWDgrBjdAN1gv9cWG3YCBCiTG6W0d9jvsD8h07AtzwGF343qF+tMD9s7wXzItMK6YXfRWuWlLzOtY9Hf4vZDlN5tdK2j36BPIfeCEyID9XhG+DdBxCTb0jpgbchqYHCgY2M+WTMhdJ6Q9B90BPpwAT9k4a9jj9UkOX45bNyVn0HGHdxen1bAD+IYpht2dfJgkJFxB4K1IDWJ694HzYMvWTTy3yecB/3VCQXchS7b6AW0n2IvgN2Bu2JO09dhjDv9JDKrCXqEvxc3Q1UZMGQg7y+6lSQU7+Q7Y4Xm86D3vO0rAbkpcbFG3CD0vBRHkNXRfn4Ob01DEc0bH1IcDZ2kXtnPbLaQdICMlbOTOAvIEugCYNTK4pzhrNRDlaBLlmU9sAlyigtdgt8IOAL4oqSF1A2gT9iEwvEv/EPQSMLsBWxL61X3nWUicxBCskzyMZ9DWgu8E3AsaJWYLQtrJKv1LOtblNeJr7uSfYykdoTPDmicecGAk8EarT7kNXqKvqgJ5xFTAPuwVvCftbmA8YBAgdMhKy2GXMmAU5tn5W2BV6FXYUNA4h+lG8QQIO/SfL4elgN9bNzv6A3L6p1Qlto66wHp4JrAq89mCQBH2Guxaqb3tlpzluEQ3rfKstPr0bW1A38Da9E1QRgA36W2BHSZW8yl3lpIDvB/g3Y0NB9OzGw/OjPYscAC1E+x7SGjbIk5serShwdssHSnGeNOHLTsuqrgOGEfi/ji1P+8s03ca735nBb/oCqJYkOvMfwxtyGp1rt3dSMOBCjyzyfrGoHf9SOgWdg+s0084KwlcAZmlioSHMX2ZZch52Mcz7HUHdC10ecTnC/sx6uO8mQregLwzSu/l2dw4KPdmwzAk3gmyY45zNAqQIzJsHmAO2NWQccr+nbagDeIe2N/0B0km9ZUkOlNBh+xpo5ald+r3+Zy6FrIJGJyysRFB3gMjYq/rPrB4wI44xOtlSia1DtlfB59Q94Bftfqc+JFlvKAPdhTy/xJ6V3V2R1X92LfS99QoCEkfeJ/z9F6Hfue5eTPIHvAzbDjYt/SL7BRJwZohj1kiSt0ewu4lhgPvq/Th6M059wXYn7Yq9DNte+Cmv4beJdp8ig6ZyrL6EDwfNoFHZgXVPUfv9L/BZoQOVEHDAvtF4yJ040ZpCZ8UMGkHuKMn2UzNb3V88oUSQkeE9BvOSo6Q5bApxHfoO54V/xp6h92lY+/qkL0e7FvhD4U+1zv+eXqvSEp9Iq4VfFLHKXg90jHeD1hcN0qQa1iDsE8h96Gv9DEwK2SjRNxOvE7fZU+U8YNfZNDKX0LvOK+IJTpqvc3PZO6DrVdk1VLO0rvW6sBu7kN+G8AV7FoHGanPAzVijMEu0U9F3yX5QhE0AcRU9yPq8bjrDmhGoh1GXyHwL/0bd38Fvd9AJzdAgwp5Djq5vVFBG8Acvn1WvpMuAvoX8P4VJujIakj8D74F3aqMtdC2sOiPq5SJRdgdkv5R4WvSxzucgyuwWh22PjAm9NtfQ+/AYfRLsZONSvuc8hbvQ/l8nt6VAvZKJiZ16n0XdkgEXRVRNoIWIO8cl7reZlIb9h0YnaU6sJcbMif7QkaEjF8QNyit3hbnTn/b59J7+m8Oi0BfWz3QtjqnP9uG/U+eVqPehqWpWCf0FfSUBZkEHYd9oM/XA+3S37JhciLoFWcMzMZOn60eQ7WfQ6vp37Pr9XzqT08D3TqtDrEQ6MgvMRkONOyzDgFrBb4SmAoYmDbKBGfCznuVHXkwxsM9EYukH+OzsNNhomiWLmjL2juFpWuwH2GfuEK/h8y1bgADw+aPgBujMWwGZcPaCjXCflJXAssLH7DVoCyFXWUDM9BOrHwOXSyq7nCQ+H+YWCrkLezaVlt0GaIdB1tEZkc4nCtjTVg/34P6vukyfqmKhEPITomxJdCFTp/2eENMbr+VDk6mkP6+ZjLipMU0h0IS82DCqI+9Az+B14AXeI4irgR6hI4yqJMDdp2C7plDV5c0QcdtyFZ2rYVcpm/TYxlmA+cCOe5+dJ1H+yod5l52F/2Vk9j4OL8y2+8w1kA7Buth/gCwc48xcchaln9DXlrAVi3of+A+xnNUYiV9DtuamKu9U6U+dCR0Sx266u6z155rERBl18/4icOuv6BJ4hkm50JmSKSDuKy36hEbOboi2/QntthKhbZ/M4T9ybYMjKFhHxpAt2zLoBTf7A94cf3xhLrsumFnFSC7GCHztXob2hZ7DJkV25OMZ/qRQj3LuGoduggyBTwr2RLzGZivAHliwQ6zcAcLyLwOu7X2yesuHU1wu1XWk1bRj31obBvQdzXhd4XN70H2ijgqeKBOPTGWFNJ7vcGuauTTohIa1L/Y4z50HUumgN9aRonXKFaHeGP3Rr2q7ydIFpI9PpZ/Ij0N+mWf6iQXRlb/KbH3S7DPd8JXhJ1VGYOsN2DDzEjfZdFNuQXcTwzGJF7GJz3mIPjQMT50dpu6d0d7SOF7h23cpf1GOfjCSGD3Jp2w/Jyt4cnSkgYs74A2HMYvpd5OiSbQSeMy4zz02cIWh12A3yV6xID7I/C0pJQYdSbWjP2/wn8NfIOzo53z2fKnlE7m3E+sy9KSLOwpz6ANUyAOAw0VyJ8a5REnWtAzzhwUCbI/ms95ZrRtVbZZYezCos3W2Cge+KFOf5MT/HG0lKmd3GXoKaTsaYAmqp5KPMmuul6vDLnE/CeWghOv0tbYivwEi7lBDeaXMCYGmgI2rTM+ZOK7sF0Zd9H9P4yesgn0eXqC/vGYs8T4MGyFlhkokQkbgrxBe0OlzQ/5z5gFCwToH1Q9xiFVYAunjvXrjL0HsLFxFtB/n05PAieoq3jfmXdB+QOdSx8rcJfCFkcReFUH3bQU8ECHhR0R/Tqw19neQWbxiqoDg7WwduBG2rYK1q/qtLuMt+27WE8l1km6w5g51gJMH3aoQwOV+WSULVEzZCwK2Ir7V6TeBD3TbwH8yLZTbdhiNqftFYkp2XoFNq2X06mZ+B/rGLtSpzgK+0/O3c1cpEAmEx1hyz7bwHyT287czuWnwbaA/WgTd2JN2A/mYEHfNIpK5BfxPeAX4FOdfn6gQU7RgO2D/QHmY+sohf7EgLlEqsSYN3COpGTjH5txKOwW6mumyjKXaO20tkFXai67g/00Ug9rXoxaNy5sthXehYUoxXjSr9DzSV5fB+fU8RSpCX0OTId9VUROoU29Av4aSw5bK0GnczoyZHORHVJh18hxLAT2lNCVhgS5wHhxQJn30fXyHCAfnnI+5JD+0opEGRr7VMAb9MExB4q+cnbi9sYb4FHwf4d5duA1+hfJ6xPQAWNvCv2JLrGBootWeVla3I1o6xHTi/TaG3diyf6ohbO2/NU4nU4aAudJIhciTn1tgHda5h2u1ZMCLbkrciViHyhwUMkReTGQpxZ5ZRzZzLkAR6t1YKgQOFqifwX6DvLZ0eewA5izZDPOuoFmYZwFNjZ0jw4+0qveO/eX+RjPbKHGLtlHtdeKA53GicKZtcNWpB3LVmawAyLY7ux869mwW/rA2iKfRxb5R57jqy3a0JWtHTH+OiF2LKsi7xDSAtcDi8MWquw+Ye139mDyw16owXjhJHl9feqQJ+iaQ86JpOwUxhi5Z1g7pzqJNUacKNCTmMMBHVEEffuweXca5Jqg/7oKXM5cCcpYm3kpOxFDCoHD3ykrWJ86Ebm+N5m/m/e2eJ/eenzrJ/Yu2+VBNjHujj1lKzmb3ftbzCUCTXjVucp4UZ0xOMheXeTVgg8gx0g3FnSe3sO7ArPosDOY9+t+7pqHpSbwdlWOcR6LFIX9IjKx6F8QGAI6gfkmwm6vz/E727TR14X9p+5mB2qvCZpgHKAKfmxIDvNoImcOHngnTwJb3JpC/2b/PkhsBNDGZhrHEgvgqTBuj0LZwZwNkSOzUUSeAGR6ZAZq2PEg91zBo+RfzyjBPhOdtKmn6Veib8RmjXSr77+Trlk8VhTYp5T5u3mTjRun695Qh4ocrpA5MvQPgO9EvqOIz26YG+vAphdxA073od725tA1PegVE/K7zcmzkaarIi+A02g/ed0R+G9PzzG+NKAP2f6sUgYeYH4A9m6C/QW2jBjTsZl/QIxWZn6hSpu/RXsSulrISVvYxIyJMa/h3fScyfGYtMxV7Gswsrp74wg7ty2rLdrgDRZhbh3m0upN4A1IbIv/MvaHtdKvV3dgp3U4Ewt6f0bZTbTmq0RIsd6cv3N/01K7CzkozMfEebP/hlfFflaKdmiU2P7XYY4q95D4XsSt2BYRup0YhHULoQGZ3KZ9xtwM4OY58EbfwzuV3rtWymbmnw+kfd7XPld+DHkRywrY2boPHT3D3uCZzGdlfF7IBOaFVLDXzF0xaX8TP27p3wGd7sBr9BNHnB6nWQaw1DjiRD7QzftkRUmFjvudReX3R7mm8qCUlLnt5CjVJQ7tiVDkU0pKy97ZVmODc3eBm+Yau+p7vkQfjvBtM5ZHHOwFkA+gVwm6I+JeN+M4AP2t0KHZPJu30DH3sxtVdkozl9c+n9yy90aHZTHAd+pehysh240Sa2LfJOBLy2Y+MOQI5UBDFvloFv1SzNngtAejQB2Os9kyR8mBzIFsLIipMlEAWm/sPkYnymYQ5xffM+8b+OPZKd5EE+Y5tgLodCVDM43Igb1KK0JhV35iOGAS8GVRob+S+o/1JoypR6T7gO3AICUa0PE4J29OH4OMvZ/bHnFuY/du/ZLQDPbhlGakDn4CP7Pv1L3ADKaw2YCjIbvnATAee2qE0IusFwlAGyI2wVxXYhXm6IK2d4wR2Tgv2FSg/TFsXWBAL1d/8Y59N7Dv2Rz3kzUz36cs/Ls69DDjtXVgvsigD3IOXDKnPxr2q8T8EvBBCXqPtUTCP6VAnrD+CBoFeHBWsplzlq9lecea7fyaSyJvKThdO2jcAo1g7YwH4D6hIvxK9D3Rpw7ei3yRc81We7CrpDge1CszhqC16Ks1hR2kCh9Bu0ys+Lq1p7xapF29xLOjSS5WVKXP3Cf2t2lPEylhXcQ7rM1hrZLKaVch/ZF2iXkD0NPgPb/k6KyX4fpZQ6XQkxY49N1H41fRcD6H/NS/rog8PYNxaZ+2FJ4TxtP+oJOB12k/2TrsrJayoZ6G3cT2xQF9vooOLNEyJNgzuAbymxiVueevw8hxvj+L/C250C0IGyrqMuc2dJZ4B/H5QEo+z8SEHMav6F8k7VmMTxpbR6dsojyDfNCDuch9t2y2HwUNB57ImxL+nwA2F2MWOOeQvgPmcoJOPmnNY8jVSe3mcTJQmfv3bSqJEt01MbPIMbc687Fk3twn90obBORiXvT5MLcusuN6JP7UJ9CLwMgtNkRh7SHr2yCTWXMDeUisB1nqaWwlGrapgeasAQJ9RbDDtk7tU9/viaX6wxbWDnyy990NVt/iUls58/n1d2Verkr9WGccpD9XdZwZ/fE6cx59YlfQJeuxZqxz2Sn07XNaXH0Wse6Ofnqt7nOqiES+BQ3K2t1f+q7QFxPmiRdgb2R8F6yPHEusD2KrauZ6Cpvem8yFbVdnXhvz9X3m4ftxXacZMj9fYbwC/2mMmUe0VTgxtyfBnnqHDLhpxTWnzMuXw6E1hs0Eub7o5fDxtLQS5fbdTG7r/aAQ3yvNa8zl96nEwOU47q0C78Bm8kReNHByH/JiJlPXgReBjBtb1iaCnmH7zTiGQE7eE2eMa1vGRtXnnhopn/p+41Kf19zTb+eEwSKXa+5NiA3Ab8w/pK8LvCfBviLv6XNfAU6yJfCYjrWzZgb4kzm59PcT0wKfzR1RD+SIGgqbdQG7T6HDzcTqAGOoD9B3WPfvufw6Va9QBlBOhCLHjnkGzCmt9wMV2IE+AGYisSYJtmIocuKlvuuE/ZC5CQrsKk3UHtjMZYP+770S2wl7ZJX4L5779NXB7u6le1uqYg+2OK85a6j+WeXh/mxEzRobsVhxLVR3kMZkDu+c/fw6L/kiX99h3XaL9crQscwbqfuSGjIvyWY97hZSpAwbCbgSfNQydqxHgJzYMudfxOqlhiRybvSqr+VjHe/mpfGtCfmHvWqxRnA7H0rmA+xU6M9iGvvIyI7M54PiDXNsJiUV/9/P5aQA64D/mXfeYz30FvajR3yJ9bMeBnZu06evXtNFbjfjVqTnEnQ/dL3I6cY+ND2RzwJ8BP0e/iHn+08qH3NnStsXPA07PtmD7Duw/l+dq1GHE9u3DvNCIpX1+LRVmK+Odas++UvEEDwFMrTNGmFiWegtEzIHcpH1jvSjAafb3ptw7GY4qAaiOdTdKZZlfSAwf5n2lu2Z+H/GMlkvXhX5QQ79YMI/5ojYs1bHenT6G0gzTeYGQP8yL2tcgE3gi3zNt8jwQRX2oBm3qyhlfo/rLYSfNxsnIm5WRC6eymeL2IsdTcQIGtj5oGXTE34Qyy4oumgLX9bE/zOvoi0Lf05k7yAPig5zaaPx6/w3wl+3DaZn7AHg/S0xMf1K+D1i3pKG9Wk6c1Dob/TpT5cc+jxgQ2FN0C8zjjXh2BNZa3UoZ8s4B9iJjS17Byiv20P6OCA/Zs8GcA/4sNyP+zuK2t/MvhVViTnwog5fthnrYQ5Viz5YxiXJQ23ZaWG9OuxZ7lkLdjRHr7DOjjl9HENjKTLfC+8S0Sf2un2jzQ+e4CitRO9diAEWWefliJxR4tNxmfQvao05niTyd5qwsZjjjvWL2lHIbBFr5XXzkPFMtaUGNus42CtBN9+9RuzzZjKAzJUmkFOd+ehqDbHB+lrGqEu4B2Mf9GlB1rLWDXyk06dllOhLAOcXbYm5iya9MwUhE+qgSa/KmlnY1uDFFn0cr8OpE2E7w36v7WUV1lktTm6V5yF0COu7gIlLwBTP7JeQyUOFPFUZQ4ONakQiF5P1w62+sJ2ZX6tYrMGgzKxAHxrADcwZqwIPAVO0DE74LjAvyAbWU8X3+t7H93tVBOZ87lrBmn1xJ0lrnXGmfs9hHib0umIxdw10LHKVaPeAb1j3Uqe+Z638uMyaXIfj2yDTFPow8B4qa9Z4JrSj6mPYGHbhlbqPExUh/yHzU/0HfAldALslb/NciR27os+BwBrgSYujLfAf7ROOcxD9MSpF+vWZRwW9HXDiqFNnbQRoiTEUb84e05wc7wp5p7/S53J134tzvpszOK5vM7CvflltKcyVKth6h7mZoF/WVvVFbgb4TmJtGPQc3o09WSYeezqzNwkn4TKfTKnPdqD/yGaN3rv3+hCr2tN9M63x38d7RLyVHmTYyK4dVUV3GeblMk4IHCAz517Yk8B/9G+q4r9JwFrPuGa+SbsGuBBrloD5w+a76fpsTN5rs5/Dhj2vgc+3SthkrwFWNbE2KIK8ZR+cHeyUCPZ9KHLOLWUL/AK6F3hP5EOL3hT02YTsV9O4Ji829iCAHGtiLy+cczTxxDMjxmRoQ89kynsH9gJ78qg6cxq5n9CrzDmTHF+z2EujJ8b1QK8VWKsN2QapQprBvmV8ChOLdbDqfNRS751Mzgqxesrjh/hom/1jtqx5FX1SImLWmUwMK/JPmFdVn4mYKdYNnUZ+MoD7oMespG6WPZtCZaOxHwsxVT27Pw4bfgpbo2ttl/YC9tQiF99iH4vChJippORtR+IfjkJgfx/avS2O2cC6JMZm6QPtM3cHeBx4wINdKWrvTFfU6OH9HfZ4guwCjc2Zk4XvbzK0JbFRbdyarQgc1NmJvIhsrXcpnsxsSELmNJyBA2x3s4l79mRpbFwSsW7PZs+OMuvogEFd5qsQQwvMzx5MEfhVb7Mf0JZ9rMkHDtctagzYf0L0O2LeFeuT37SHk7D5NLFM3aF8KfaPelK1t3wuYyPAE7IKGWwzn1syQNPMo6e/RmEenUTbG7qpLHIgYb8oVg+4zwYNMFeVMVrssSXq3N+8l+k75PxOwEn0k9oiF2bOPDPWKZQ04iPqvDp5A7YWaN6hfBZ5031f8ThahXXscd8zTvBVRH41aDRpW5i0cXTHSS0sR0MmLdzuRqWJaH+pAW8r9UrS5i2JqxUO/nwLZ83rFBGHrwhZoNTi+w4zLQOPe6S92P8K+GT/XNFLZb+2e0Fni340iPAsd9/PpYjrN5keMJk4SdL/q/g77nnoQ6Nyrbw+bUsIupkUMn2nmtAHJbYhrWzVzHux7s0ZVAuQh8lIKXO/X2yOvF932C+C/paiN5J5uBc+/37fe3h1y7vZQzC51M8VhmqfStNIjPdcz1QwAsDhMoAwEwFIAADR13RQwotmQP3+ujA/phkK78q1SYOj4g2IOWMEhp0dDukrlMyCHYphiKadeWEIqPfjsJk1clwosPU0NOL+rLeXPx+XzA03kwlOF9e0UIvj2yoYzNwlAcoX1p8kcF9+h7OEerRGAIat5Aza63G2CX7WiS0FTyw2Od7f6dlr4n09d3+xR80bjwLDkW52578vuhJnhCPAwa0TjF35EKB1z3zvZO/O3uOwH+6lteeZLvMecf/fHC0aJ316jwOzJ98/3Z/k74eAdDKs4209gM8+56hPcb4P8T5wH4O+TNONUjXC97egPSbY3JPnJta2kL2PuGaBzwM2s9lS4BRya16IuWAi6XpE/jgnACGohqQ30Ucz28MYgA/nkVyfdskW4CqX2F3IXdc59AWNG0zge8ahN2vnmUUOdiaxd3BwEt8nhvTzKL3Hgr1j+3LcaCHuccs+qc6+N3MnNQ5zMkvPOFnZbTxO5LiB8dhcEJS3b1PH7GEvU+d1VuF0dw+xoM/tS0yTSbDHGBUEXbiHTu5v60Wa/PGSbAbADw69WBd21PUqa6VWZEMyCXyWyhA3ae7xAJqwKN/23/EUXF9OgYQL8A8QLjrNN3P39tprdXf2OuNPboiaDMzZO1W4t9/2cuK1QOP8tY4T0/CWzePoaMkr/Izijmkq22P4rYDj2nd27/hOdPSdo37IeF/olGyPZSO8KQnQRIdM7j3ZYMLckOfTAVGZa1kgkgVBFwHIoWd1hwZtME5kneIe9fYFrzlS05sa7PG7Jfh55HklkzGSaw5O9PzZV9yuW8463p8pK0Hj8R5xvvupAy7l1TWMcerheD9azR0d9mzOAjkC0MUexb18j/iWmU2GLNCwPDKW83ISumg4YAKy2cSz/DgBct8H2T/utWzjHWA8iv7GE4G14oZqXQsGRWv7PBUOgg7n1BWhs9OEUuFYcG4TZ10AwG+pSWNTIVejMfgoNdadEPoF74h7cg3e5CCTk8Q8YTTUpgYLcJrF0W1Gfu3XCVyW6IUk4Zq9oUWz4qShUi74B/qAfhH9mxlYwH43o4noE130eHZj9vKvFXZKvS2xkcI+QA+awV54w5bodx3ppQnONmlk17rxJnEf7Kg7iJuvYS+hEwpubVH4/loZ+zR+WE7/dbodhstgehCwhVhwtkwqpMJd0mH5xy2HOjfZZXEBAvF/gJjqPUHkJQDQx3Gpyqg9FR2tvt/zXZUmOLTeFxyaGhNHvGgAES+ZaBQ/o2WWReelQebzu6KIJmWeFxJEgNAPz4KiPT3k42eyEr4/HzVVKFYHygkH4gd+LLi3ooqZ1RWjBYQELcpdMXlOO5fplllHPIEpBlRrjpNJuj1slLuTtao2ACKVrHlYa5a44jWKJuadeFKdHwC8FDtOosQHvQIsGxzuv/v/AZA7C26SPAsB")


In [ ]:
_COMPRESSED = base64.b64decode("".join(_PAYLOAD_CHUNKS), validate=True)
if hashlib.sha256(_COMPRESSED).hexdigest() != "f6c064d3d50b2a2584b6d3e8e84f704255c9852e00701c93c09f4d990c3c48fa":
    raise ValueError("Embedded package fingerprint mismatch; no authentication attempted.")
_FILES = json.loads(gzip.decompress(_COMPRESSED))
_ROOT = Path(tempfile.gettempdir()) / "furusato-preview30-f6c064d3d50b2a25"
for _relative, _encoded in _FILES.items():
    _path = (_ROOT / _relative).resolve()
    if _ROOT.resolve() not in _path.parents:
        raise ValueError("Unsafe embedded package path.")
    _raw = base64.b64decode(_encoded, validate=True)
    _path.parent.mkdir(parents=True, exist_ok=True)
    if _path.exists() and _path.read_bytes() != _raw:
        raise ValueError("Existing package bytes changed; use a clean driver session.")
    _path.write_bytes(_raw)
sys.path.insert(0, str(_ROOT / "tools/provisioning"))
sys.path.insert(0, str(_ROOT / "tools/data-agent"))
for _module, _directory in {'preview30_runtime': 'tools/provisioning', 'preview30_deploy': 'tools/provisioning', 'preview30_verify': 'tools/provisioning', 'preview30_ontology': 'tools/provisioning', 'preview30_agent': 'tools/provisioning', 'preview30_activation': 'tools/provisioning', 'preview30_compatibility': 'tools/provisioning', 'v3_artifacts': 'tools/provisioning', 'workshop_runtime': 'tools/provisioning', 'reference_kql': 'tools/provisioning', 'activation_runtime': 'tools/provisioning', 'source_grounded_profile': 'tools/data-agent', 'answer_contract_profile': 'tools/data-agent', 'time_layer_isolation': 'tools/data-agent', 'standard_contract_restoration': 'tools/data-agent', 'fresh_grounded_profile': 'tools/data-agent', 'native_evaluation': 'tools/data-agent', 'native_mcp': 'tools/data-agent', 'measured_contract_profile': 'tools/data-agent'}.items():
    if _module in sys.modules and Path(sys.modules[_module].__file__).resolve().parent != (_ROOT / _directory).resolve():
        del sys.modules[_module]
print("Sealed portable package loaded. No Fabric mutation has occurred.")


In [ ]:
if ACTION not in ('preflight', 'deploy-sources', 'verify-sources', 'deploy-realtime', 'deploy-core', 'handoff-ontology', 'verify-gold', 'deploy-semantic-model', 'refresh-model', 'verify-model', 'deploy-agent', 'publish-agent'):
    raise ValueError("Action is outside this notebook's bounded role. Use the CLI for importing notebooks.")
from pathlib import Path
from types import SimpleNamespace
import json
import shutil
import preview30_runtime as rt
import preview30_deploy as deploy
import preview30_verify as verify

context = notebookutils.runtime.context
if ENVIRONMENT not in ("dev", "test", "prod") or not SCOPE_FILE or not PRIVATE_EVIDENCE_DIR:
    raise ValueError("Set explicit environment, private scope file and private evidence directory. No cloud call made.")
scope_document = rt.load(Path(SCOPE_FILE))
scope = rt.scope_from_document(scope_document, ENVIRONMENT)
if scope["participantId"] != PARTICIPANT_ID:
    raise ValueError("Participant parameter differs from the private frozen scope.")
if scope["workspaceId"] != context["currentWorkspaceId"] or scope["workspaceName"] != context["currentWorkspaceName"]:
    raise ValueError("Notebook context differs from the private frozen workspace.")
if not EXPECTED_WORKSPACE_NAME or EXPECTED_WORKSPACE_NAME != context["currentWorkspaceName"]:
    raise ValueError("Set the exact expected workspace name before execution.")

class NotebookCredential:
    def get_token(self, scope_uri):
        audiences = {
            "https://api.fabric.microsoft.com/.default": "pbi",
            "https://storage.azure.com/.default": "storage",
            "https://kusto.kusto.windows.net/.default": "kusto",
            "https://analysis.windows.net/powerbi/api/.default": "pbi",
        }
        if scope_uri not in audiences:
            raise ValueError("Unsupported token audience.")
        return SimpleNamespace(token=notebookutils.credentials.getToken(audiences[scope_uri]))

credential = NotebookCredential()
evidence = rt.private_directory(Path(PRIVATE_EVIDENCE_DIR))
client = rt.EvidenceClient(scope, evidence, credential=credential)
current_notebook = client.request("GET", "/workspaces/" + scope["workspaceId"] + "/items/" + context["currentNotebookId"]).json()
if current_notebook.get("folderId") != scope["folderId"]:
    raise ValueError("The current Notebook is outside the explicitly selected folder.")

if ACTION == "preflight":
    if APPLY_CHANGES:
        raise ValueError("Preflight is read-only; APPLY_CHANGES must be False.")
    prefix = "/workspaces/" + scope["workspaceId"]
    inventory = {
        "workspace": client.request("GET", prefix).json(),
        "items": client.paged(prefix + "/items"),
        "folders": client.paged(prefix + "/folders"),
        "recoverableItems": client.paged(prefix + "/recoverableItems"),
    }
    baseline = rt.load(rt.PREVIEW / "provisioning/baseline-hashes.json")
    for relative, expected_sha in baseline["files"].items():
        path = rt.BASE / relative
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha:
            raise ValueError("Extracted baseline byte mismatch: " + relative)
    state_file = evidence / "deployment-state.json"
    owned_state = rt.load(state_file) if state_file.exists() else None
    plan = rt.build_plan(scope, inventory, baseline, owned_state)
    rt.save(evidence / "resource-plan.json", plan)
    rt.save(evidence / "inventory-preflight.json", inventory)
    RESULT = {"mode": "read-only-preflight", "planSha256": plan["planSha256"],
              "blockers": plan["blockers"], "folderMappingVerified": scope["folderMappingVerified"],
              "cloudMutationPerformed": False}
else:
    read_only = ACTION.startswith("verify-") or ACTION == "preview-metadata"
    if not read_only and not APPLY_CHANGES:
        raise ValueError("No write made. Review the private plan, then explicitly enable APPLY_CHANGES.")
    if not read_only and context.get("isForPipeline", False) and not ALLOW_AUTOMATED_APPLY:
        raise ValueError("Automated mutation is not explicitly enabled.")
    if PLAN_FILE:
        external_plan = rt.load(Path(PLAN_FILE))
        rt.save(evidence / "resource-plan.json", external_plan)
    args = SimpleNamespace(
        environment=ENVIRONMENT, scope=Path(SCOPE_FILE), evidence_dir=evidence,
        write_gate=Path(WRITE_GATE_FILE) if WRITE_GATE_FILE else None,
        confirm=CONFIRMED_PLAN_SHA256, run_notebook01=RUN_NOTEBOOK01,
        expected_definition_sha256=EXPECTED_DEFINITION_SHA256,
        rule_statement_overrides=RULE_STATEMENT_OVERRIDES, credential=credential,
    )
    actions = {
        "deploy-sources": deploy.deploy_sources, "deploy-notebooks": deploy.deploy_notebooks,
        "verify-sources": verify.verify_sources, "deploy-realtime": deploy.deploy_realtime,
        "deploy-core": deploy.deploy_core, "preview-metadata": deploy.metadata_change,
        "apply-metadata": lambda a: deploy.metadata_change(a, apply=True),
        "handoff-ontology": deploy.handoff_ontology, "verify-gold": verify.verify_gold,
        "deploy-semantic-model": deploy.deploy_semantic_model, "verify-model": verify.verify_model,
        "refresh-model": deploy.refresh_model_once,
        "deploy-agent": deploy.deploy_agent,
        "publish-agent": deploy.publish_agent_candidate,
    }
    if ACTION not in actions:
        raise ValueError("Unsupported action; no private endpoint fallback.")
    RESULT = actions[ACTION](args)
print(json.dumps(RESULT, ensure_ascii=False, indent=2))
